##Camada Silver: Limpeza e Transformação

Serão aplicadas transformações e limpeza de dados na camada Silver. Haverá verificação de integridade de valores nulos e de chaves. 
Incialmente, foi gerada uma tabela flat que foram desmembradas em um esquema floco de neve, para tratamento de arrays e nulos.
A tabela fato fato_netflix será particionada por ano de lançamento de filmes / seriados para melhorar o desempenho de leitura e escrita.

### Imports necessários

In [0]:
# Importar as bibliotecas necessárias

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.functions import col, trim, regexp_extract, to_date, when, split
from pyspark.sql.functions import explode, row_number, lit
from pyspark.sql.window import Window
from pyspark.sql.functions import dayofmonth, month, date_format, year, quarter, dayofweek
from delta.tables import DeltaTable
import gc



### Limpeza dos dados em cache, remoção e recriação de pastas da camada silver

In [0]:
# Limpar todos os dados em cache

spark.catalog.clearCache()

# clearCache() limpa o cache de todos os objetos em cache no SparkSession atual, liberando uma quantidade significativa de memória quando múltiplos DataFrames estão sendo reutilizados.


gc.collect()

#Comentário: Esse comando força o coletor de lixo a executar imediatamente, liberando a memória de objetos Python que não estão mais em uso.

Out[63]: 404

In [0]:
silver_path = "/mnt/netflix/silver"

In [0]:
dbutils.fs.rm("/mnt/netflix/silver",recurse=True)
dbutils.fs.mkdirs(silver_path)

Out[65]: True

###Iniciar a sessão Spark

In [0]:
# Iniciar a SparkSession com configurações otimizadas
spark = SparkSession.builder \
    .appName("Transformação Data Silver") \
    .config("spark.sql.shuffle.partitions", "4")  \
    .config("spark.sql.files.maxPartitionBytes", "128MB") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

# Define um número fixo de partições para shuffle, melhorando o paralelismo 
# Geralmente, o número de partições corresponde a 2 * numero_cores
# num_particoes = 2 * numero_cores
# O cluster configurado no Databricks Community Edition contém 2 cores.                
# Define o tamanho máximo de partições para evitar muitos arquivos pequenos.      
# Habilita otimizações adaptativas, ajustando o número de partições dinamicamente com base no tamanho dos dados



###Criar o database silver 

In [0]:
%sql CREATE DATABASE IF NOT EXISTS silver;

In [0]:
%sql SHOW DATABASES;

databaseName
bronze
default
silver


### Obter a tabela netflix_titles da camada bronze 

Obter a tabela netflix_titles da camada bronze e fazer as transformações. Inserir a coluna de data de ingestão. Inicialmente, será criada uma tabela flat. 

####Geração da tabela flat com acréscimo de data de ingestão e tratamento de dados

Nesta sessão será feita a ingestão da tabela oriunda da camada bronze, com acréscimo de data de ingestão, criação de arrays para colunas com valores múltiplos, por exemplo: país, gênero, diretor, cast (ator).

Também será feito o tratamento da duração. Se for um filme ("Movie), a duração será em minutos. Se for um seriado("TV Show"), a duração será em temporadas.

A coluna "date_added" (data cadastrado na Netflix), inicialmente no formato string, será convertida para o formato data.

In [0]:
# Ler dados da bronze
silver_df = spark.read.table("bronze.netflix_titles")

# Transformações de limpeza
silver_clean = (silver_df
  .withColumn("title", trim(col("title")))
  .withColumn("director", when(col("director") == "", None).otherwise(col("director")))
  .withColumn("cast", when(col("cast") == "", None).otherwise(col("cast")))
  .withColumn("country", when(col("country") == "", None).otherwise(col("country")))
  .withColumn("date_added", 
              to_date(trim(col("date_added")), "MMMM d, yyyy"))
  .withColumn("duration_minutes",
              when(col("type") == "Movie",
                   regexp_extract(col("duration"), r"(\d+)", 1).cast("int"))
              .otherwise(None))
  .withColumn("duration_seasons",
              when(col("type") == "TV Show",
                   regexp_extract(col("duration"), r"(\d+)", 1).cast("int"))
              .otherwise(None))
  .withColumn("director_array", split(col("director"), ",\s*"))
  .withColumn("country_array", split(col("country"), ",\s*"))
  .withColumn("genre_array", split(col("listed_in"), ",\s*"))
  .withColumn("cast_array", split(col("cast"), ",\s*"))
)

# Gera uma tabela flat de limpeza, com o acréscimo da coluna ingestion_timestamp
# A coluna ingestion_timestamp representa a data/hora de ingestão dos dados 

silver_clean = (silver_clean
  .withColumn("ingestion_timestamp", current_timestamp())
  .select(
      "show_id",
      "type",
      "title",
      "director",
      "cast",
      "country",
      "date_added",
      "release_year",
      "rating",
      "duration",
      "duration_minutes",
      "duration_seasons",
      "listed_in",
      "description",
      "director_array",
      "country_array",
      "genre_array",
      "cast_array",
      "ingestion_timestamp"
  ))

# Salvar na camada silver
(silver_clean.write
  .format("delta")
  .mode("overwrite")
  .option("mergeSchema", "true")
  .save(silver_path))

# Criar tabela silver flat
spark.sql(f"""
CREATE TABLE IF NOT EXISTS silver.netflix_titles_clean
USING DELTA
LOCATION '{silver_path}'
""")




Out[69]: DataFrame[]

###Esquema Floco de Neve

A tabela flat netflix_titles_clean transformação da tabela flat em um esquema floco de neve, para melhor tratamento de nulos e arrays.

### Criar tabelas de dimensões



In [0]:
%sql
-- deleta a tabela silver.dim_date para recriá-la, pois o código está apresentando erro
DROP TABLE IF EXISTS silver.dim_date;

In [0]:

# Cria a tabela dim_tempo
silver_clean_df = spark.read.table("silver.netflix_titles_clean")
tabela_dim_date = "dim_date"




# Preencher dimensão tempo a partir dos dados
dates_df = silver_clean_df.select("date_added").distinct().filter(col("date_added").isNotNull())

dim_date_df = (dates_df
    .withColumn("day", dayofmonth("date_added"))
    .withColumn("month", month("date_added"))
    .withColumn("month_name", date_format("date_added", "MMMM"))
    .withColumn("year", year("date_added"))
    .withColumn("quarter", quarter("date_added"))
    .withColumn("day_of_week", dayofweek("date_added"))
    .withColumn("day_name", date_format("date_added", "EEEE"))
    .withColumn("is_weekend", (dayofweek("date_added").isin(1, 7))))

dim_date_df.show()

spark.sql("SHOW TABLES IN silver").show()

# Escrever usando API DeltaTable
DeltaTable.createIfNotExists(spark)\
    .tableName("silver.dim_date") \
    .addColumns(dim_date_df.schema) \
    .partitionedBy("year") \
    .location(f"{silver_path}/{tabela_dim_date}") \
    .execute()
    
dim_date_df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("silver.dim_date")


+----------+---+-----+----------+----+-------+-----------+---------+----------+
|date_added|day|month|month_name|year|quarter|day_of_week| day_name|is_weekend|
+----------+---+-----+----------+----+-------+-----------+---------+----------+
|2021-09-20| 20|    9| September|2021|      3|          2|   Monday|     false|
|2021-09-19| 19|    9| September|2021|      3|          1|   Sunday|      true|
|2021-08-27| 27|    8|    August|2021|      3|          6|   Friday|     false|
|2021-08-25| 25|    8|    August|2021|      3|          4|Wednesday|     false|
|2021-08-23| 23|    8|    August|2021|      3|          2|   Monday|     false|
|2021-08-21| 21|    8|    August|2021|      3|          7| Saturday|      true|
|2021-08-20| 20|    8|    August|2021|      3|          6|   Friday|     false|
|2021-08-19| 19|    8|    August|2021|      3|          5| Thursday|     false|
|2021-08-18| 18|    8|    August|2021|      3|          4|Wednesday|     false|
|2021-08-15| 15|    8|    August|2021|  

In [0]:
   spark.sql("DESCRIBE silver.dim_date").show()

+--------------------+---------+-------+
|            col_name|data_type|comment|
+--------------------+---------+-------+
|          date_added|     date|   null|
|                 day|      int|   null|
|               month|      int|   null|
|          month_name|   string|   null|
|                year|      int|   null|
|             quarter|      int|   null|
|         day_of_week|      int|   null|
|            day_name|   string|   null|
|          is_weekend|  boolean|   null|
|# Partition Infor...|         |       |
|          # col_name|data_type|comment|
|                year|      int|   null|
+--------------------+---------+-------+



In [0]:
%sql USE silver;

In [0]:
%sql SELECT * FROM silver.dim_date;

date_added,day,month,month_name,year,quarter,day_of_week,day_name,is_weekend
2019-12-31,31,12,December,2019,4,3,Tuesday,false
2019-12-28,28,12,December,2019,4,7,Saturday,true
2019-12-26,26,12,December,2019,4,5,Thursday,false
2019-12-21,21,12,December,2019,4,7,Saturday,true
2019-12-19,19,12,December,2019,4,5,Thursday,false
2019-12-18,18,12,December,2019,4,4,Wednesday,false
2019-12-12,12,12,December,2019,4,5,Thursday,false
2019-12-09,9,12,December,2019,4,2,Monday,false
2019-12-08,8,12,December,2019,4,1,Sunday,true
2019-12-03,3,12,December,2019,4,3,Tuesday,false


In [0]:
%sql
-- deleta a tabela silver.dim_tipo_conteudo para recriá-la, pois o código está apresentando erro (sem este comando)
DROP TABLE IF EXISTS silver.dim_tipo_conteudo;

In [0]:


# Cria a tabela dim_tipo_conteudo
silver_clean_df = spark.read.table("silver.netflix_titles_clean")
tabela_dim_tipo_conteudo= "dim_tipo_conteudo"




# Preencher dimensão tipo de conteudo a partir dos dados
tipo_conteudo = content_types = [
    (1, "Movie", "Feature films and movies"),
    (2, "TV Show", "Television series and shows")
]

dim_tipo_conteudo_df = spark.createDataFrame(tipo_conteudo, ["content_type_id", "content_type", "description"])

dim_tipo_conteudo_df.show()

# Escrever usando API DeltaTable
DeltaTable.createIfNotExists(spark) \
    .tableName("silver.dim_tipo_conteudo") \
    .addColumns(dim_tipo_conteudo_df.schema) \
    .location(f"{silver_path}/{tabela_dim_tipo_conteudo}") \
    .execute()
    
dim_tipo_conteudo_df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("silver.dim_tipo_conteudo")


+---------------+------------+--------------------+
|content_type_id|content_type|         description|
+---------------+------------+--------------------+
|              1|       Movie|Feature films and...|
|              2|     TV Show|Television series...|
+---------------+------------+--------------------+



In [0]:
%sql SELECT * FROM dim_tipo_conteudo;

content_type_id,content_type,description
2,TV Show,Television series and shows
1,Movie,Feature films and movies


In [0]:
%sql
-- deleta a tabela silver.dim_diretor para recriá-la, pois o código está apresentando erro (sem este comando)
DROP TABLE IF EXISTS silver.dim_diretor;

####Tabela Dimensão diretor

Tabela Dimensão diretor, com tratamento para múltiplos diretores (array de diretores)

In [0]:

# Cria a tabela dim_diretor
silver_clean_df = spark.read.table("silver.netflix_titles_clean")
tabela_dim_diretor= "dim_diretor"

dim_diretor_df = (silver_clean_df
  .withColumn("director", explode(split(col("director"), ",\s*")))
  .select("director")
  .distinct()
  .filter(col("director").isNotNull())
  .withColumn("director_id", row_number().over(Window.orderBy("director"))))


dim_diretor_df.show()

# Escrever usando API DeltaTable
DeltaTable.createIfNotExists(spark) \
    .tableName("silver.dim_diretor") \
    .addColumns(dim_diretor_df.schema) \
    .location(f"{silver_path}/{tabela_dim_diretor}") \
    .execute()
    
dim_diretor_df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("silver.dim_diretor")


+--------------------+-----------+
|            director|director_id|
+--------------------+-----------+
|         A. L. Vijay|          1|
|        A. Raajdheep|          2|
|           A. Salaam|          3|
|     A.R. Murugadoss|          4|
|     Aadish Keluskar|          5|
|        Aamir Bashir|          6|
|          Aamir Khan|          7|
|          Aanand Rai|          8|
|         Aaron Burns|          9|
|        Aaron Hancox|         10|
|          Aaron Hann|         11|
|        Aaron Lieber|         12|
|      Aaron Moorhead|         13|
|           Aaron Nee|         14|
|        Aaron Sorkin|         15|
|       Aaron Woodley|         16|
|         Aaron Woolf|         17|
|     Aatmaram Dharne|         18|
|      Abba T. Makama|         19|
|Abbas Alibhai Bur...|         20|
+--------------------+-----------+
only showing top 20 rows



In [0]:
%sql SELECT * FROM dim_diretor;

director,director_id
A. L. Vijay,1
A. Raajdheep,2
A. Salaam,3
A.R. Murugadoss,4
Aadish Keluskar,5
Aamir Bashir,6
Aamir Khan,7
Aanand Rai,8
Aaron Burns,9
Aaron Hancox,10


In [0]:
%sql
-- deleta a tabela silver.dim_ator para recriá-la, pois o código está apresentando erro (sem este comando)
DROP TABLE IF EXISTS silver.dim_ator;

####Tabela dimensão de Atores

Tabela Dimensão de atores, com tratamento para múltiplos atores (array de atores)

In [0]:

# Cria a tabela dim_ator
silver_clean_df = spark.read.table("silver.netflix_titles_clean")
tabela_dim_ator= "dim_ator"

dim_ator_df = (silver_clean_df
  .withColumn("cast", explode(split(col("cast"), ",\s*")))
  .select("cast")
  .distinct()
  .filter(col("cast").isNotNull())
  .withColumn("ator_id", row_number().over(Window.orderBy("cast"))))


dim_ator_df.show()

# Escrever usando API DeltaTable
DeltaTable.createIfNotExists(spark) \
    .tableName("silver.dim_ator") \
    .addColumns(dim_ator_df.schema) \
    .location(f"{silver_path}/{tabela_dim_ator}") \
    .execute()
    
dim_ator_df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("silver.dim_ator")


+--------------------+-------+
|                cast|ator_id|
+--------------------+-------+
|"Riley" Lakdhar D...|      1|
|        'Najite Dede|      2|
|            2 Chainz|      3|
|                2Mex|      4|
|             4Minute|      5|
|             50 Cent|      6|
|                9m88|      7|
|A Boogie Wit tha ...|      8|
|      A. Murat Özgen|      9|
|       A.C. Peterson|     10|
|          A.D. Miles|     11|
|           A.J. Cook|     12|
|        A.J. Johnson|     13|
|       A.J. LoCascio|     14|
|         A.K. Hangal|     15|
|         A.R. Rahman|     16|
|     A.S. Sasi Kumar|     17|
|              AC Lim|     18|
|                AFRA|     19|
|            AJ Bowen|     20|
+--------------------+-------+
only showing top 20 rows



In [0]:
%sql
-- deleta a tabela silver.dim_pais para recriá-la, pois o código está apresentando erro (sem este comando)
DROP TABLE IF EXISTS silver.dim_pais;

####Tabela Dimensão de Países
Tabela Dimensão de países, com tratamento para múltiplos países (array de países)

In [0]:



# Cria a tabela dim_pais
silver_clean_df = spark.read.table("silver.netflix_titles_clean")
tabela_dim_pais= "dim_pais"

dim_pais_df = (silver_clean_df
  .withColumn("country", explode(split(col("country"), ",\s*")))
  .select("country")
  .distinct()
  .filter(col("country").isNotNull())
  .withColumn("country_id", row_number().over(Window.orderBy("country"))))


dim_pais_df.show()

# Escrever usando API DeltaTable
DeltaTable.createIfNotExists(spark) \
    .tableName("silver.dim_pais") \
    .addColumns(dim_pais_df.schema) \
    .location(f"{silver_path}/{tabela_dim_pais}") \
    .execute()
    
dim_pais_df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("silver.dim_pais")



+------------+----------+
|     country|country_id|
+------------+----------+
|            |         1|
| Afghanistan|         2|
|     Albania|         3|
|     Algeria|         4|
|      Angola|         5|
|   Argentina|         6|
|     Armenia|         7|
|   Australia|         8|
|     Austria|         9|
|  Azerbaijan|        10|
|     Bahamas|        11|
|  Bangladesh|        12|
|     Belarus|        13|
|     Belgium|        14|
|     Bermuda|        15|
|    Botswana|        16|
|      Brazil|        17|
|    Bulgaria|        18|
|Burkina Faso|        19|
|    Cambodia|        20|
+------------+----------+
only showing top 20 rows



In [0]:
%sql
SELECT * FROM silver.dim_pais;


country,country_id
,1
Afghanistan,2
Albania,3
Algeria,4
Angola,5
Argentina,6
Armenia,7
Australia,8
Austria,9
Azerbaijan,10


In [0]:
%sql
-- deleta a tabela silver.dim_genero para recriá-la, pois o código está apresentando erro (sem este comando)
DROP TABLE IF EXISTS silver.dim_genero;

####Criar tabela dimensão de gêneros de filmes

A tabela dimensão dim_genero será criada para o tratamento de array de gênero de títulos, ou seja, consiste na normalização da tabela de títulos.

In [0]:

# Cria a tabela dim_genero - gênero de filmes
silver_clean_df = spark.read.table("silver.netflix_titles_clean")
tabela_dim_genero= "dim_genero"

dim_genero_df = (silver_clean_df
  .withColumn("genre", explode(split(col("listed_in"), ",\s*")))
  .select("genre")
  .distinct()
  .filter(col("genre").isNotNull())
  .withColumn("genre_id", row_number().over(Window.orderBy("genre"))))



dim_genero_df.show()

# Escrever usando API DeltaTable
DeltaTable.createIfNotExists(spark) \
    .tableName("silver.dim_genero") \
    .addColumns(dim_genero_df.schema) \
    .location(f"{silver_path}/{tabela_dim_genero}") \
    .execute()
    
dim_genero_df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("silver.dim_genero")


+--------------------+--------+
|               genre|genre_id|
+--------------------+--------+
|  Action & Adventure|       1|
|      Anime Features|       2|
|        Anime Series|       3|
|    British TV Shows|       4|
|Children & Family...|       5|
|   Classic & Cult TV|       6|
|      Classic Movies|       7|
|            Comedies|       8|
|      Crime TV Shows|       9|
|         Cult Movies|      10|
|       Documentaries|      11|
|          Docuseries|      12|
|              Dramas|      13|
|Faith & Spirituality|      14|
|       Horror Movies|      15|
|  Independent Movies|      16|
|International Movies|      17|
|International TV ...|      18|
|            Kids' TV|      19|
|     Korean TV Shows|      20|
+--------------------+--------+
only showing top 20 rows



In [0]:
%sql SELECT * FROM silver.dim_genero;

genre,genre_id
Action & Adventure,1
Anime Features,2
Anime Series,3
British TV Shows,4
Children & Family Movies,5
Classic & Cult TV,6
Classic Movies,7
Comedies,8
Crime TV Shows,9
Cult Movies,10


In [0]:
%sql
-- deleta a tabela silver.fato_netflix para recriá-la, pois o código está apresentando erro (sem este comando)
DROP TABLE IF EXISTS silver.fato_netflix;

%md
###Criação da Tabela Fato Netflix

In [0]:

# Cria a tabela fato_netflix
silver_clean_df = spark.read.table("silver.netflix_titles_clean")
tabela_fato_netflix = "fato_neflix"


# Preparar dados da tabela fato <==
fato_netflix_df = (silver_clean_df
  .join(dim_tipo_conteudo_df, silver_clean_df["type"] == dim_tipo_conteudo_df["content_type"], "left")
  .select(
      "show_id",
      "title",
      "content_type_id",
      "date_added",
      "release_year",
      "duration_minutes",
      "duration_seasons",
      "rating",
      "silver.netflix_titles_clean.description"
  ))



# Escrever usando API DeltaTable
DeltaTable.createIfNotExists(spark) \
    .tableName("silver.fato_netflix") \
    .addColumns(fato_netflix_df.schema) \
    .location(f"{silver_path}/{tabela_fato_netflix}") \
    .execute()
    

fato_netflix_df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("silver.fato_netflix")


In [0]:
%sql SELECT * FROM silver.fato_netflix;

show_id,title,content_type_id,date_added,release_year,duration_minutes,duration_seasons,rating,description
s1,Dick Johnson Is Dead,1,2021-09-25,2020,90,null,PG-13,"As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable."
s2,Blood & Water,2,2021-09-24,2021,null,2,TV-MA,"After crossing paths at a party, a Cape Town teen sets out to prove whether a private-school swimming star is her sister who was abducted at birth."
s3,Ganglands,2,2021-09-24,2021,null,1,TV-MA,"To protect his family from a powerful drug lord, skilled thief Mehdi and his expert team of robbers are pulled into a violent and deadly turf war."
s4,Jailbirds New Orleans,2,2021-09-24,2021,null,1,TV-MA,"Feuds, flirtations and toilet talk go down among the incarcerated women at the Orleans Justice Center in New Orleans on this gritty reality series."
s5,Kota Factory,2,2021-09-24,2021,null,2,TV-MA,"In a city of coaching centers known to train India’s finest collegiate minds, an earnest but unexceptional student and his friends navigate campus life."
s6,Midnight Mass,2,2021-09-24,2021,null,1,TV-MA,"The arrival of a charismatic young priest brings glorious miracles, ominous mysteries and renewed religious fervor to a dying town desperate to believe."
s7,My Little Pony: A New Generation,1,2021-09-24,2021,91,null,PG,"Equestria's divided. But a bright-eyed hero believes Earth Ponies, Pegasi and Unicorns should be pals — and, hoof to heart, she’s determined to prove it."
s8,Sankofa,1,2021-09-24,1993,125,null,TV-MA,"On a photo shoot in Ghana, an American model slips back in time, becomes enslaved on a plantation and bears witness to the agony of her ancestral past."
s9,The Great British Baking Show,2,2021-09-24,2021,null,9,TV-14,"A talented batch of amateur bakers face off in a 10-week competition, whipping up their best dishes in the hopes of being named the U.K.'s best."
s10,The Starling,1,2021-09-24,2021,104,null,PG-13,A woman adjusting to life after a loss contends with a feisty bird that's taken over her garden — and a husband who's struggling to find a way forward.


In [0]:
%sql SELECT * FROM silver.dim_pais;

country,country_id
,1
Afghanistan,2
Albania,3
Algeria,4
Angola,5
Argentina,6
Armenia,7
Australia,8
Austria,9
Azerbaijan,10


####Criar tabelas de junção para relacionamentos muitos-para-muitos

Estas tabelas são utilizadas para tratamento de arrays e de relacionamentos muitos-para_muitos. Consiste em normalizar as tabelas, para melhorar o tratamento de nulos e duplicatas.

In [0]:
%sql
-- deleta a tabela silver.conteudo_diretor_junc para recriá-la, pois o código está apresentando erro (sem este comando)
DROP TABLE IF EXISTS silver.conteudo_diretor_junc;

####Tabela de junção Conteúdo-Diretor

In [0]:
# Conteúdo-Diretor


# Cria a tabela de junção conteúdo-diretor (relacionamento muito para muitos)
silver_clean_df = spark.read.table("silver.netflix_titles_clean")
tabela_conteudo_diretor = "conteudo_diretor_junc"

conteudo_diretor_df = (silver_clean_df
  .withColumn("director", explode(split(col("director"), ",\s*")))
  .join(dim_diretor_df, "director")
  .select("show_id", "director_id"))
 

# Escrever usando API DeltaTable
DeltaTable.createIfNotExists(spark) \
    .tableName("silver.conteudo_diretor_junc") \
    .addColumns(conteudo_diretor_df.schema) \
    .location(f"{silver_path}/{tabela_conteudo_diretor}") \
    .execute()
    

conteudo_diretor_df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("silver.conteudo_diretor_junc")

In [0]:
%sql
-- deleta a tabela silver.conteudo_ator_junc para recriá-la, pois o código está apresentando erro (sem este comando)
DROP TABLE IF EXISTS silver.conteudo_ator_junc;

####Tabela de junção Conteúdo-Ator

In [0]:
# Conteúdo-Ator


# Cria a tabela de junção conteúdo-diretor (relacionamento muito para muitos)
silver_clean_df = spark.read.table("silver.netflix_titles_clean")
tabela_conteudo_ator = "conteudo_ator_junc"

conteudo_ator_df = (silver_clean_df
  .withColumn("cast", explode(split(col("cast"), ",\s*")))
  .join(dim_ator_df, "cast")
  .select("show_id", "ator_id"))
 

# Escrever usando API DeltaTable
DeltaTable.createIfNotExists(spark) \
    .tableName("silver.conteudo_ator_junc") \
    .addColumns(conteudo_ator_df.schema) \
    .location(f"{silver_path}/{tabela_conteudo_ator}") \
    .execute()
    

conteudo_ator_df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("silver.conteudo_ator_junc")

In [0]:
%sql SELECT * FROM silver.conteudo_ator_junc

show_id,ator_id
s2138,1
s2196,2
s2103,2
s4962,3
s1328,4
s5998,5
s8465,6
s8315,6
s7986,6
s7869,6


In [0]:
%sql SELECT * FROM silver.conteudo_diretor_junc;

show_id,director_id
s1,2533
s3,2323
s6,3163
s7,3917
s7,2290
s8,1646
s9,335
s10,4583
s12,2544
s13,873


In [0]:
%sql
-- deleta a tabela silver.conteudo_pais_junc para recriá-la, pois o código está apresentando erro (sem este comando)
DROP TABLE IF EXISTS silver.conteudo_pais_junc;

####Tabela de junção Conteúdo-País

Relacionamento muito para muitos. Um país pode produzir muitos títulos(conteúdos) e um título(conteúdo) pode ser produzido em muitos países.

In [0]:
# Conteúdo-País


# Cria a tabela de junção conteúdo-país (relacionamento muito para muitos)

silver_clean_df = spark.read.table("silver.netflix_titles_clean")
tabela_conteudo_pais = "conteudo_pais_junc"


conteudo_pais_df = (silver_clean_df
  .withColumn("country", explode(split(col("country"), ",\s*")))
  .join(dim_pais_df, "country")
  .select("show_id", "country_id"))
 


# Escrever usando API DeltaTable
DeltaTable.createIfNotExists(spark) \
    .tableName("silver.conteudo_pais_junc") \
    .addColumns(conteudo_pais_df.schema) \
    .location(f"{silver_path}/{tabela_conteudo_pais}") \
    .execute()
    

conteudo_pais_df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("silver.conteudo_pais_junc")

In [0]:
%sql SELECT * FROM silver.conteudo_pais_junc;

show_id,country_id
s1,117
s2,101
s5,47
s8,117
s8,41
s8,19
s8,116
s8,40
s8,36
s9,116


In [0]:
%sql
-- deleta a tabela silver.conteudo_genero_junc para recriá-la, pois o código está apresentando erro (sem este comando)
DROP TABLE IF EXISTS silver.conteudo_genero_junc;

####Tabela de junção Conteúdo-Gênero (muito para muitos)

Relacionamento muito para muitos. Um título (conteúdo) pode ser classificado de vários gêneros diferentes e pode haver um título (conteúdo) com vários gêneros.

In [0]:
# Conteúdo-Gênero


# Cria a tabela de junção conteúdo-gênero (relacionamento muito para muitos)

silver_clean_df = spark.read.table("silver.netflix_titles_clean")
tabela_conteudo_genero = "conteudo_genero_junc"


conteudo_genero_df = (silver_clean_df
  .withColumn("genre", explode(split(col("listed_in"), ",\s*")))
  .join(dim_genero_df, "genre")
  .select("show_id", "genre_id"))
 

# Escrever usando API DeltaTable
DeltaTable.createIfNotExists(spark) \
    .tableName("silver.conteudo_genero_junc") \
    .addColumns(conteudo_genero_df.schema) \
    .location(f"{silver_path}/{tabela_conteudo_genero}") \
    .execute()
    

conteudo_genero_df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("silver.conteudo_genero_junc")

In [0]:
%sql SELECT * FROM silver.conteudo_genero_junc;

show_id,genre_id
s1,11
s2,18
s2,35
s2,37
s3,9
s3,18
s3,33
s4,12
s4,24
s5,18


##Qualidade dos dados

###Valores nulos críticos

Verificar integridade das chaves.

Obs: O Databricks Community Edition não permite o uso de constraints (PK e FK) em SQL


In [0]:
%sql
-- Valores nulos críticos
-- Verificar integridade de chaves
SELECT 
SUM(CASE WHEN show_id IS NULL THEN 1 ELSE 0 END) AS quant_nulos_show_id,
SUM(CASE WHEN title IS NULL THEN 1 ELSE 0 END) AS quant_nulos_titulos,
SUM(CASE WHEN date_added IS NULL THEN 1 ELSE 0 END) AS quant_nulos_date_added,
  (SELECT COUNT(*) FROM silver.fato_netflix) AS total_titulos,
ROUND((quant_nulos_date_added * 100 /total_titulos),6) AS percent_nulos_date_added

FROM silver.fato_netflix;


quant_nulos_show_id,quant_nulos_titulos,quant_nulos_date_added,total_titulos,percent_nulos_date_added
0,0,10,8807,0.113546


In [0]:
%sql
-- Verifica se existem diretores duplicados

SELECT director, COUNT(*) as quant_diretores_duplicados FROM silver.dim_diretor as diretor_duplicados GROUP BY 1 HAVING COUNT(*) > 1;

director,quant_diretores_duplicados


###Controle de órfãos

In [0]:
%sql
-- controle de órfãos
-- Atores sem conteúdo associado -> orphaned_actors
SELECT 
    COUNT(*) AS orphaned_actors
FROM silver.dim_ator a
LEFT JOIN silver.conteudo_ator_junc j ON a.ator_id = j.ator_id
WHERE j.ator_id IS NULL;



orphaned_actors
0


In [0]:
%sql
-- problema: Gêneros duplicados
-- solução: Implementar processo de normalização de gêneros	

SELECT genre, COUNT(*) as genero_duplicado FROM dim_genero GROUP BY 1 HAVING COUNT(*) > 1;

genre,genero_duplicado



#### Controle de órfãos

1. Controle de datas: verifica se há inserção de datas futuras na tabela dim_date
2. Controle de órfãos - verifica se existediretor sem conteúdo associado

In [0]:
%sql
-- Problema: Datas futuras
-- Solução: Adicionar validação no pipeline
SELECT * FROM dim_date WHERE date_added > CURRENT_DATE();

date_added,day,month,month_name,year,quarter,day_of_week,day_name,is_weekend


####Contole de órfãos - diretor sem conteúdo associado

In [0]:
%sql
-- controle de órfãos
-- Diretores sem conteúdo associado -> orphaned_directors
SELECT 
    COUNT(*) AS orphaned_directors
FROM silver.dim_diretor d
LEFT JOIN silver.conteudo_diretor_junc j ON d.director_id = j.director_id
WHERE j.director_id IS NULL;



orphaned_directors
0


In [0]:
%sql
-- Conteúdo sem gênero
SELECT 
    COUNT(DISTINCT f.show_id) AS conteudo_sem_genero
FROM silver.fato_netflix f
LEFT JOIN silver.conteudo_genero_junc j ON f.show_id = j.show_id
WHERE j.show_id IS NULL;

conteudo_sem_genero
0


####Conteúdo com diretor associado

Conteúdo com diretor associado -  quantidade de conteúdo com diretor associado (diretor não nulo)

In [0]:
%sql
-- controle de órfãos
-- Conteudo com diretores associados 
SELECT 
    COUNT(DISTINCT(f.show_id)) AS conteudo_com_diretor
FROM silver.fato_netflix f
RIGHT JOIN silver.conteudo_diretor_junc j ON f.show_id = j.show_id;



conteudo_com_diretor
6173


### Contagem total de títulos, diretor com nulos e contéudo com diretor

Esta consulta no esquema floco de neve torna-se mais complexa do que na tabela flat porque requer join e subconsutas.

In [0]:
%sql
-- controle de órfãos
-- Conteudo com diretores associados (conteudo_com_diretor) 
-- Conteúdo sem diretor associado (conteudo_sem_diretor)
SELECT 
    COUNT(DISTINCT(f.show_id)) AS conteudo_com_diretor,
    (SELECT COUNT(*) FROM silver.fato_netflix) AS total_titulos,
    (total_titulos - conteudo_com_diretor) AS conteudo_sem_diretor,
    ROUND(conteudo_sem_diretor * 100 /total_titulos, 4) AS perc_sem_diretor
FROM silver.fato_netflix f
RIGHT JOIN silver.conteudo_diretor_junc j ON f.show_id = j.show_id


conteudo_com_diretor,total_titulos,conteudo_sem_diretor,perc_sem_diretor
6173,8807,2634,29.908


In [0]:
%sql
-- controle de órfãos
-- Conteúdo com atores associados (conteudo_com_ator) 
-- Conteúdo sem ator associado (conteudo_sem_ator)
SELECT 
    COUNT(DISTINCT(f.show_id)) AS conteudo_com_ator,
    (SELECT COUNT(*) FROM silver.fato_netflix) AS total_titulos,
    (total_titulos - conteudo_com_ator) AS conteudo_sem_ator,
    ROUND(conteudo_sem_ator *100 /total_titulos, 4) AS perc_sem_ator
FROM silver.fato_netflix f
RIGHT JOIN silver.conteudo_ator_junc j ON f.show_id = j.show_id


conteudo_com_ator,total_titulos,conteudo_sem_ator,perc_sem_ator
7982,8807,825,9.3675


In [0]:
%sql
-- controle de órfãos
-- Conteudo com países associados (conteudo_com_pais) 
-- Conteúdo sem país associado (conteudo_sem_pais)
SELECT 
    COUNT(DISTINCT(f.show_id)) AS conteudo_com_pais,
    (SELECT COUNT(*) FROM silver.fato_netflix) AS total_titulos,
    (total_titulos - conteudo_com_pais) AS conteudo_sem_pais,
    ROUND(conteudo_sem_pais * 100 /total_titulos, 4) AS perc_sem_pais
FROM silver.fato_netflix f
RIGHT JOIN silver.conteudo_pais_junc j ON f.show_id = j.show_id

conteudo_com_pais,total_titulos,conteudo_sem_pais,perc_sem_pais
7976,8807,831,9.4357


In [0]:
%sql
-- Problema: Países não padronizados
-- Recomendação: Criar dicionário de países oficiais e eliminar valores em branco para país
SELECT DISTINCT country FROM silver.dim_pais
WHERE country IS NOT NULL AND country <> '';

country
Albania
Algeria
Argentina
Australia
Austria
Belarus
Burkina Faso
Cameroon
Ecuador
Ghana


In [0]:
%sql
-- eliminar as linhas com país em branco na tabela dim_pais
DELETE FROM silver.dim_pais WHERE country IS NOT NULL AND country = '';

num_affected_rows
1


In [0]:
%sql 
SELECT DISTINCT(count(country)) FROM silver.dim_pais;

count(country)
122


In [0]:
%sql
SELECT * FROM silver.dim_pais;

country,country_id
Afghanistan,2
Albania,3
Algeria,4
Angola,5
Argentina,6
Armenia,7
Australia,8
Austria,9
Azerbaijan,10
Bahamas,11


In [0]:
%sql 
SHOW TABLES IN silver;


database,tableName,isTemporary
silver,conteudo_ator_junc,false
silver,conteudo_diretor_junc,false
silver,conteudo_genero_junc,false
silver,conteudo_pais_junc,false
silver,dim_ator,false
silver,dim_date,false
silver,dim_diretor,false
silver,dim_genero,false
silver,dim_pais,false
silver,dim_tipo_conteudo,false


In [0]:
# Listar todas as tabelas no banco de dados silver

tables_df = spark.sql("SHOW TABLES IN silver")
# Iterar sobre cada tabela e descrever
for row in tables_df.collect():
        table_name = row.tableName    
        display((f"Descrevendo tabela: {table_name}"))    
        display(spark.sql(f"DESCRIBE silver.{table_name}"))

'Descrevendo tabela: conteudo_ator_junc'

col_name,data_type,comment
show_id,string,null
ator_id,int,null


'Descrevendo tabela: conteudo_diretor_junc'

col_name,data_type,comment
show_id,string,null
director_id,int,null


'Descrevendo tabela: conteudo_genero_junc'

...

col_name,data_type,comment
content_type_id,bigint,null
content_type,string,null
description,string,null


'Descrevendo tabela: fato_netflix'

col_name,data_type,comment
show_id,string,null
title,string,null
content_type_id,bigint,null
date_added,date,null
release_year,int,null
duration_minutes,int,null
duration_seasons,int,null
rating,string,null
description,string,null


'Descrevendo tabela: netflix_titles_clean'

col_name,data_type,comment
show_id,string,null
type,string,null
title,string,null
director,string,null
cast,string,null
country,string,null
date_added,date,null
release_year,int,null
rating,string,null
duration,string,null


###Limpeza dos dados em cache e coleta de lixo

In [0]:
# Limpar todos os dados em cache

spark.catalog.clearCache()

# clearCache() limpa o cache de todos os objetos em cache no SparkSession atual, liberando uma quantidade significativa de memória quando múltiplos DataFrames estão sendo reutilizados.


gc.collect()

#Comentário: Esse comando força o coletor de lixo a executar imediatamente, liberando a memória de objetos Python que não estão mais em uso.

Out[122]: 797